# Computational DAG

The computational DAG is screamer's capstone: it lets you define a
multi-input, multi-step computation **once** as a graph of `Input` nodes,
alignment operators, and functors - then run that exact same graph in
**batch mode** (over historical arrays for backtesting) or in **live mode**
(event-by-event for production) and get byte-identical results either way.
The graph is compiled and executed entirely in C++, so even graphs with
many nodes have near-zero Python overhead.  This notebook defines a
spread-smoothing signal over two asynchronous price feeds, builds a `Dag`,
runs it both ways, and asserts the outputs match.

In [ ]:
import numpy as np
from screamer import Input, Dag, RollingMean, Sub, combine_latest

## Defining the graph

A graph is built by composing `Node` objects.  `Input(name)` creates a
named source placeholder.  `combine_latest` aligns two async feeds into one
2-column stream (causal forward-fill carry).  Functors like `Sub()` and
`RollingMean` operate on nodes just as they do on arrays - the result is
another node, not a computed value.  Actual computation happens only when
you call the `Dag` object.

Use C++ functors (`Sub()`, `Add()`, etc.) for arithmetic inside the graph
rather than Python lambdas - they integrate natively into the C++ execution
engine.

In [ ]:
# Define inputs: named placeholders for the two async price feeds
a = Input("a")
b = Input("b")

# Align the two feeds: emit a row whenever either ticks, carrying last-seen
# values (emit="when_all" default: wait until both feeds have been seen)
aligned = combine_latest(a, b)

# Compute the spread: Sub() takes aligned[:, 0] - aligned[:, 1] per row
spread = Sub()(aligned)

# Smooth the spread with a rolling mean (window=10)
signal = RollingMean(10)(spread)

# Build the DAG: define which nodes are inputs and which are outputs
dag = Dag(inputs=[a, b], outputs=[signal])

print("Graph: a, b -> combine_latest -> Sub() -> RollingMean(10) -> signal")
print("dag inputs: [a, b]")
print("dag outputs: [signal]")

## Feed data: two async series with different tick rates

Feed `a` ticks at odd keys, feed `b` at even keys - no shared timestamps.
Together they produce 13 aligned events (after `when_all` warm-up) which
the DAG processes in one shot (batch) or event-by-event (live).

In [ ]:
# Two async series with different timestamps
fa = (np.array([1, 3, 5, 7, 9, 11, 13], dtype=np.int64), np.arange(7.0))
fb = (np.array([2, 4, 6, 8, 10, 12],    dtype=np.int64), np.arange(6.0) * 2)

print("feed a - keys:", fa[0], "  values:", fa[1])
print("feed b - keys:", fb[0], "  values:", fb[1])

## Running the DAG: batch vs live

`dag(*feeds)` runs the entire graph over the input arrays in one C++ pass -
the **batch** (backtest) path.  `dag.stream(*feeds)` runs the same graph
event-by-event - the **live** (streaming) path.  Both return
`(keys, values)` tuples.  The two results must be byte-identical: this is
the batch==live guarantee at the DAG level.

In [ ]:
# Batch path: whole arrays in, whole array out
batch = dag(fa, fb)
bk, bv = batch

# Live path: same graph, same data, event-by-event
live = dag.stream(fa, fb)
sk, sv = live

print("batch keys  :", bk)
print("batch values:", bv)
print()
print("live  keys  :", sk)
print("live  values:", sv)

## Asserting batch == live

The core regression test: if batch and live ever diverge, the DAG has a
causality or state-management bug.  `assert_array_equal` treats NaN as
equal, so warm-up NaNs are handled correctly.

In [ ]:
# Keys must be identical
np.testing.assert_array_equal(bk, sk)

# Values must be byte-identical (NaN == NaN in this comparison)
np.testing.assert_array_equal(bv, sv)

print("batch == live:", np.array_equal(bv, sv, equal_nan=True))
print()
print("Non-NaN signal values:")
mask = ~np.isnan(bv)
for k, v in zip(bk[mask], bv[mask]):
    print(f"  key={k:3d}  signal={v:.4f}")

## Extending the graph

A DAG can have multiple outputs, and graphs compose: the output of one
`Dag` call can be fed as input to another computation.  Here we add a
second output - the raw spread (pre-smoothing) - to show multi-output DAGs.

In [ ]:
# Multi-output DAG: expose both the raw spread and the smoothed signal
dag2 = Dag(inputs=[a, b], outputs=[spread, signal])

raw_stream, smooth_stream = dag2(fa, fb)
rk, rv = raw_stream
sk2, sv2 = smooth_stream

print("raw spread   :", rv)
print("smoothed (10):", sv2)

# The smoothed output must still match the single-output DAG
np.testing.assert_array_equal(bv, sv2)
print()
print("multi-output dag2 smooth == single-output dag: PASSED")

**Takeaways**

- `Input(name)` creates a named source placeholder; compose it with
  `combine_latest`, `Sub()`, `RollingMean`, etc. to build a graph.
- `Dag(inputs=[...], outputs=[...])` compiles the graph; calling
  `dag(*feeds)` runs it in batch and `dag.stream(*feeds)` runs it live -
  same code, byte-identical results.
- The entire graph executes in C++: Python overhead is limited to the
  call boundary, not per-event.
- Use C++ functors (`Sub()`, `Add()`, `Mul()`, etc.) for arithmetic inside
  the graph - they integrate into the native execution path.
- `Dag` can have multiple outputs; each output is an independent
  `(keys, values)` pair with its own key axis.
- Define once, run batch or live: this is the library's central thesis.